In [1]:
import random
from datetime import datetime, timedelta, timezone
from typing import Iterator

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
from faker import Faker
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType


In [2]:
fake = Faker("ko_KR")
random.seed(42)


def make_articles(n: int) -> list[dict]:
    now = datetime.now(timezone.utc)
    subjects = ["경제", "사회", "정치", "IT", "국제", "스포츠", "문화"]
    keywords = ["AI", "반도체", "환율", "증시", "부동산", "스타트업", "노동", "교육", "기후", "금리", "물가"]
    return [
        {
            "published": (now - timedelta(hours=random.randint(0, 240), minutes=random.randint(0, 59))).isoformat(),
            "subject": random.choice(subjects),
            "keyword": random.choice(keywords),
            "title": fake.sentence(nb_words=10),
            "summary": " ".join(fake.paragraphs(nb=2)),
            "description": " ".join(fake.paragraphs(nb=6)),
            "original_link": fake.url(),
            "link": fake.url(),
            "created_at": now.isoformat(),
        }
        for _ in range(n)
    ]


def get_elastic_search(hosts: list, basic_auth: tuple[str, str]):
    return Elasticsearch(hosts=hosts, basic_auth=basic_auth, verify_certs=False)


@F.udf(StringType())
def norm_text(s: str) -> str:
    if s is None:
        return None
    return s.strip()


def to_iso(v):
    # Spark Timestamp -> Python datetime -> ISO string
    if v is None:
        return None
    try:
        return v.isoformat()
    except Exception:
        return str(v)


def index_partition(rows: Iterator, hosts: list, basic_auth: tuple[str, str], index_name: str, bulk_size: int) -> Iterator[int]:
    client = Elasticsearch(hosts=hosts, basic_auth=basic_auth, verify_certs=False)
    batch = []
    indexed = 0

    def flush():
        nonlocal batch, indexed
        if not batch:
            return
        bulk(client, batch, raise_on_error=True, refresh=False)
        indexed += len(batch)
        batch = []

    for r in rows:
        d = r.asDict(recursive=False)
        doc_id = d.get("id")
        d["published"] = to_iso(d.get("published"))
        d["created_at"] = to_iso(d.get("created_at"))
        batch.append({"_op_type": "index", "_index": index_name, "_id": str(doc_id) if doc_id is not None else None, "_source": d})

        if len(batch) >= bulk_size:
            flush()

    flush()
    yield indexed


def swap_alias(es: Elasticsearch, alias: str, new_index: str):
    old_indices = []
    if es.indices.exists_alias(name=alias):
        old_indices = list(es.indices.get_alias(name=alias).keys())
    actions = [{"remove": {"index": old, "alias": alias}} for old in old_indices]
    actions.append({"add": {"index": new_index, "alias": alias}})
    es.indices.update_aliases(actions=actions)
    print(f"[alias] {alias} -> {new_index} (removed: {old_indices})")


In [3]:
mysql_host = "mysql-primary"
mysql_port = 3306
mysql_db = "mmix"
mysql_user = "mmix"
mysql_pass = "mmix"
mysql_database = "mmix"
table = "news_articles"
driver = "com.mysql.cj.jdbc.Driver"
jdbc_url = f"jdbc:mysql://{mysql_host}:{mysql_port}/{mysql_db}?useUnicode=true&characterEncoding=utf8&useSSL=false&serverTimezone=UTC"
properties = {"user": mysql_user, "password": mysql_pass, "driver": driver}

elasticsearch_base_index = "news_articles"
elasticsearch_alias = "news_articles_current"
elasticsearch_index_name = f"{elasticsearch_base_index}-{datetime.now().strftime('%Y%m%d%H%M%S')}"
elasticsearch_hosts = ["http://elasticsearch:9200"]
elasticsearch_basic_auth = ("elastic", "elastic")
elasticsearch_bulk_size = 1000

index_body = {
    "settings": {"number_of_shards": 1, "number_of_replicas": 0},
    "mappings": {
        "properties": {
            "id": {"type": "long"},
            "published": {"type": "date"},
            "subject": {"type": "text", "fields": {"keyword": {"type": "keyword", "ignore_above": 256}}},
            "keyword": {"type": "keyword", "ignore_above": 256},
            "title": {"type": "text", "fields": {"keyword": {"type": "keyword", "ignore_above": 256}}},
            "summary": {"type": "text"},
            "description": {"type": "text"},
            "original_link": {"type": "keyword", "ignore_above": 1024},
            "link": {"type": "keyword", "ignore_above": 1024},
            "created_at": {"type": "date"},
        }
    },
}

In [4]:
spark = SparkSession.builder.appName("ElasticSearch Example").master("spark://localhost:7077").getOrCreate()

26/01/09 17:37:53 WARN Utils: Your hostname, genius.local resolves to a loopback address: 127.0.0.1; using 10.12.2.51 instead (on interface en0)
26/01/09 17:37:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/09 17:37:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
spark.createDataFrame(make_articles(100000)).write.mode("append").jdbc(url=jdbc_url, table=table, properties=properties)
spark.createDataFrame(make_articles(100000)).write.mode("append").jdbc(url=jdbc_url, table=table, properties=properties)
spark.createDataFrame(make_articles(100000)).write.mode("append").jdbc(url=jdbc_url, table=table, properties=properties)

26/01/09 17:38:07 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/01/09 17:38:07 WARN TaskSetManager: Stage 0 contains a task of very large size (25097 KiB). The maximum recommended task size is 1000 KiB.
26/01/09 17:38:25 WARN TaskSetManager: Stage 1 contains a task of very large size (25072 KiB). The maximum recommended task size is 1000 KiB.
26/01/09 17:38:41 WARN TaskSetManager: Stage 2 contains a task of very large size (25081 KiB). The maximum recommended task size is 1000 KiB.


In [6]:
bounds = spark.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("query", "SELECT MIN(id) AS lower_bound, MAX(id) AS upper_bound FROM news_articles") \
    .options(**properties) \
    .load() \
    .first()

lower_bound, upper_bound = int(bounds["lower_bound"]), int(bounds["upper_bound"]) + 1
print(lower_bound, upper_bound)

1 900001


In [7]:
dataframe = spark.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", table) \
    .option("partitionColumn", "id") \
    .option("lowerBound", lower_bound) \
    .option("upperBound", upper_bound) \
    .option("numPartitions", 16) \
    .options(**properties) \
    .load()

dataframe.count()

900000

In [8]:
# predicates = [
#     "id BETWEEN 1 AND 250000",
#     "id BETWEEN 250001 AND 500000",
#     "id BETWEEN 500001 AND 750000",
#     "id BETWEEN 750001 AND 1000000",
# ]
#
# predicates_dataframe = spark.read.jdbc(url=jdbc_url, table="news_articles", predicates=predicates, properties=properties)
# predicates_dataframe.count()

In [9]:
elasticsearch = get_elastic_search(elasticsearch_hosts, ("elastic", "elastic"))
if elasticsearch.indices.exists(index=elasticsearch_index_name):
    elasticsearch.indices.delete(index=elasticsearch_index_name)

elasticsearch.indices.create(index=elasticsearch_index_name, **index_body)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'news_articles-20260109173752'})

In [10]:
norm_text_dataframe = dataframe.withColumn("title", norm_text("title")).withColumn("summary", norm_text("summary")).withColumn("description", norm_text("description")).withColumn("subject", norm_text("subject"))
counts = norm_text_dataframe.repartition(8).rdd.mapPartitions(lambda it: index_partition(it, elasticsearch_hosts, elasticsearch_basic_auth, elasticsearch_index_name, elasticsearch_bulk_size)).collect()
print("partition counts:", counts, "total:", sum(counts))

partition counts: [112499, 112498, 112499, 112499, 112498, 112503, 112504, 112500] total: 900000


In [11]:
swap_alias(elasticsearch, elasticsearch_alias, elasticsearch_index_name)

[alias] news_articles_current -> news_articles-20260109173752 (removed: [])


In [12]:
response = elasticsearch.search(index=elasticsearch_alias, size=5, query={"match_all": {}}, sort=[{"published": {"order": "desc"}}])
for hit in response["hits"]["hits"]:
    s = hit["_source"]
    print(s["id"], s["published"], s["subject"], s["title"])

810374 2026-01-09T08:38:29 경제 Animi perferendis mollitia officia aliquam vitae corporis aspernatur veniam ex.
880465 2026-01-09T08:38:29 정치 Et rem dolorum assumenda voluptate quas aspernatur nihil ea quia iste asperiores in nam.
896554 2026-01-09T08:38:29 문화 Mollitia explicabo molestias eius illum debitis eius consequuntur nesciunt vero.
806565 2026-01-09T08:38:29 사회 Quidem aspernatur corporis debitis necessitatibus quod corrupti neque consectetur.
878634 2026-01-09T08:38:29 국제 Nihil rerum deserunt numquam nesciunt quae dolore.


In [13]:
spark.stop()